## Load Land Regions for Country
Read the preprocessed region-area Parquet file, filter to country land regions, and convert the WKB geometry column into a GeoDataFrame.

In [2]:
from pathlib import Path

import pandas as pd
import geopandas as gpd

repo_root = Path.cwd().resolve().parents[1]
region_area_path = repo_root / 'data' / 'results' / 'region_area.parquet'

region_df = pd.read_parquet(region_area_path)
country_filter = region_df['country'].astype(str).str.startswith('IL')
land_filter = region_df['is_land'].astype(bool)
filtered_df = region_df[country_filter & land_filter].copy()

regions_gdf = gpd.GeoDataFrame(
    filtered_df.drop(columns='geometry'),
    geometry=gpd.GeoSeries.from_wkb(filtered_df['geometry'], crs='EPSG:4326')
)

print(f'Loaded {len(regions_gdf)} region features with country=DE and is_land=TRUE')


Loaded 6 region features with country=DE and is_land=TRUE


## Derive Bounding Box
Validate that `regions_gdf` exists, compute its overall bounding box, and show the bounds as both text and a Shapely polygon.

In [3]:
from shapely.geometry import box

if 'regions_gdf' not in locals():
    raise RuntimeError('regions_gdf is not defined. Run the previous cell first.')

regions_bbox = None
if regions_gdf.empty:
    print('regions_gdf is empty; no bounding box to compute.')
else:
    minx, miny, maxx, maxy = regions_gdf.total_bounds
    regions_bbox = (float(minx), float(miny), float(maxx), float(maxy))
    bbox_geom = box(minx, miny, maxx, maxy)
    print(
        'Bounding box (minx, miny, maxx, maxy): '
        f'({minx:.6f}, {miny:.6f}, {maxx:.6f}, {maxy:.6f})'
    )
    bbox_geom


Bounding box (minx, miny, maxx, maxy): (34.267499, 29.487789, 35.684354, 33.290418)


## Building source summary

Leverages the helper function to stream every Overture buildings parquet file, assign each building to its containing region via spatial filtering, add per-dataset counters, and persist the enriched `regions_gdf` to a GeoParquet file.

In [ ]:
import importlib
import logging
import sys

if 'regions_gdf' not in locals():
    raise RuntimeError('regions_gdf is not defined. Run the earlier cells first.')
if 'regions_bbox' not in locals() or regions_bbox is None:
    raise RuntimeError('regions_bbox is not defined. Run the bounding-box cell first.')

module_root = repo_root / 'code' / 'overture_analysis'
if str(module_root) not in sys.path:
    sys.path.append(str(module_root))

building_region_counter = importlib.import_module('functions.building_region_counter')
building_region_counter = importlib.reload(building_region_counter)
summarise_buildings_by_source = building_region_counter.summarise_buildings_by_source

buildings_base_path = repo_root / 'gis_data' / 'overturemaps-us-west-2' / 'release' / '2025-08-20.1' / 'theme=buildings' / 'type=building'
results_dir = repo_root / 'data' / 'results'
processing_dir = results_dir / 'processing'
results_dir.mkdir(parents=True, exist_ok=True)
processing_dir.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger('building_summary')
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False

summary_file_paths = summarise_buildings_by_source(
    regions_gdf,
    buildings_base_path=buildings_base_path,
    batch_size=50000,
    logger=logger,
    filter_bbox=regions_bbox,
    processing_dir=processing_dir,
)
print(f'Generated {len(summary_file_paths)} partial files in {processing_dir}')


## Merge per-file summaries

Combine the worker outputs stored in the processing directory into the final regions GeoDataFrame.

In [4]:
import importlib
import logging
import sys
from pathlib import Path

if 'regions_gdf' not in locals():
    raise RuntimeError('regions_gdf is not defined. Run the earlier cells first.')

processing_dir = repo_root / 'data' / 'results' / 'processing'
if 'summary_file_paths' not in locals() or not summary_file_paths:
    summary_file_paths = sorted(processing_dir.glob('*_summary.geoparquet'))
    if not summary_file_paths:
        raise RuntimeError('No summary files found in the processing directory.')

module_root = repo_root / 'code' / 'overture_analysis'
if str(module_root) not in sys.path:
    sys.path.append(str(module_root))

building_region_counter = importlib.import_module('functions.building_region_counter')
building_region_counter = importlib.reload(building_region_counter)
merge_region_summaries = building_region_counter.merge_region_summaries

logger = logging.getLogger('building_summary')
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False

results_dir = repo_root / 'data' / 'results'
results_dir.mkdir(parents=True, exist_ok=True)
output_path = results_dir / 'ge_buildings_summary.geoparquet'

regions_with_building_counts = merge_region_summaries(
    regions_gdf,
    summary_file_paths,
    logger=logger,
)

regions_with_building_counts.to_parquet(output_path, index=False)
print(f'Saved building source summary to {output_path}')


2025-11-17 10:18:22,060 - INFO - Merged counts from part-00056-c8f91a12-b93f-4285-9103-a7779634c4ce-c000.zstd_summary.geoparquet
2025-11-17 10:18:22,116 - INFO - Merged counts from part-00110-c8f91a12-b93f-4285-9103-a7779634c4ce-c000.zstd_summary.geoparquet
2025-11-17 10:18:22,170 - INFO - Merged counts from part-00111-c8f91a12-b93f-4285-9103-a7779634c4ce-c000.zstd_summary.geoparquet


Saved building source summary to /workspaces/micromamba_cuda/data/results/ge_buildings_summary.geoparquet


In [5]:
import pandas as pd

if 'regions_with_building_counts' not in locals():
    raise RuntimeError('regions_with_building_counts is not defined. Run the merge cell first.')

geometry_name = regions_with_building_counts.geometry.name
base_columns = {'region_id', 'name', geometry_name}

# Heuristic: dataset count columns should be integer-like and not part of the base metadata.
dataset_columns = [
    col for col in regions_with_building_counts.columns
    if col not in base_columns and regions_with_building_counts[col].dtype.kind in {'i', 'u'}
]

if not dataset_columns:
    raise RuntimeError('No dataset count columns found in regions_with_building_counts.')

non_zero_mask = (regions_with_building_counts[dataset_columns] > 0).any(axis=1)
non_zero_df = regions_with_building_counts.loc[
    non_zero_mask,
    ['region_id', 'name', *dataset_columns]
].copy()

if non_zero_df.empty:
    print('No regions have non-zero building counts for the tracked datasets.')
else:
    display(non_zero_df)


KeyError: "['region_id', 'name'] not in index"